In [4]:
import numpy as np
from aeon.classification import BaseClassifier
from aeon.transformations.collection.convolution_based import MultiRocket
from aeon.transformations.collection.convolution_based._hydra import HydraTransformer
from aeon.utils.validation import check_n_jobs
from aeon.transformations.collection.interval_based import QUANTTransformer
import numpy as np
import polars as pl
from aeon.classification.base import BaseClassifier
from aeon.classification.feature_based import (
    Catch22Classifier,
)
import os
from aeon.transformations.collection.convolution_based import Rocket
from aeon.datasets.tsc_datasets import univariate
from sklearn.base import clone
from aeon.classification.convolution_based import MultiRocketHydraClassifier
from aeon.classification.convolution_based import RocketClassifier
from sklearn.metrics import accuracy_score
from aeon.classification.interval_based import QUANTClassifier
from autotsc import utils, models
from tqdm import tqdm
from aeon.classification.feature_based import Catch22Classifier
from aeon.classification.interval_based import QUANTClassifier
from tqdm import tqdm
from aeon.benchmarking import resampling
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from aeon.classification.interval_based import RSTSF
from joblib import parallel_backend

2025-12-29 20:12:48.627633: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/petelin/AutoTSC/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
dataset = 'SmallKitchenAppliances'
X_train, y_train, X_test, y_test = utils.load_dataset(dataset)

In [3]:
e = RSTSF(n_estimators=500, n_jobs=-1)
v = e.fit_predict_proba(X_train, y_train)
y_pred = e.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc:.4f}")

Accuracy on SmallKitchenAppliances: 0.8347


In [4]:
e = RSTSF(n_estimators=500, n_jobs=-1)
e.fit(X_train, y_train)
y_pred = e.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc:.4f}")

Accuracy on SmallKitchenAppliances: 0.8347


In [5]:
e = RSTSF(n_estimators=500, n_jobs=1)
e.fit(X_train, y_train)
y_pred = e.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc:.4f}")

Accuracy on SmallKitchenAppliances: 0.8320


In [6]:
e = RSTSF(n_estimators=500, n_jobs=8)
e.fit(X_train, y_train)
y_pred = e.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc:.4f}")

Accuracy on SmallKitchenAppliances: 0.8320


In [7]:
from joblib import Parallel, delayed

def run_one(run_id, X_train, y_train, X_test, y_test):
    e = RSTSF(n_estimators=500, n_jobs=1)
    e.fit(X_train, y_train)
    y_pred = e.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return run_id, acc

n_runs = 10

results = Parallel(n_jobs=-1)(
    delayed(run_one)(i, X_train, y_train, X_test, y_test) 
    for i in range(n_runs)
)

for run_id, acc in results:
    print(f"Run {run_id}: {acc:.4f}")

Run 0: 0.8293
Run 1: 0.8267
Run 2: 0.8213
Run 3: 0.8320
Run 4: 0.8320
Run 5: 0.8213
Run 6: 0.8267
Run 7: 0.8347
Run 8: 0.8267
Run 9: 0.8267


In [1]:
from sklearn.datasets import make_classification
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from joblib import Parallel, delayed
import time
from threadpoolctl import threadpool_limits
from sklearn.linear_model import RidgeClassifierCV

In [10]:
X, y = make_classification(
    n_samples=10000,
    n_features=1000,
    n_informative=100,
    n_redundant=100,
    random_state=42
)

In [ ]:
e = ExtraTreesClassifier(n_estimators=500, n_jobs=2)
e.fit(X, y)
y_pred = e.predict(X)
acc = accuracy_score(y, y_pred)
print(f"Accuracy with n_jobs=1: {acc:.4f}")

Accuracy with n_jobs=1: 1.0000


In [12]:
e = ExtraTreesClassifier(n_estimators=500, n_jobs=8)
e.fit(X, y)
y_pred = e.predict(X)
acc = accuracy_score(y, y_pred)
print(f"Accuracy with n_jobs=8: {acc:.4f}")

Accuracy with n_jobs=8: 1.0000


In [13]:
e = ExtraTreesClassifier(n_estimators=500, n_jobs=-1)
e.fit(X, y)
y_pred = e.predict(X)
acc = accuracy_score(y, y_pred)
print(f"Accuracy with n_jobs=-1: {acc:.4f}")

Accuracy with n_jobs=-1: 1.0000


In [7]:
X, y = make_classification(
    n_samples=5000,
    n_features=20000,
    n_informative=100,
    n_redundant=100,
    random_state=42
)

In [8]:
e = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
e.fit(X, y)
y_pred = e.predict(X)
acc = accuracy_score(y, y_pred)
print(f"Accuracy with {acc:.4f}")

Accuracy with 1.0000


In [10]:
e = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
with threadpool_limits(limits=4):
    e.fit(X, y)
y_pred = e.predict(X)
acc = accuracy_score(y, y_pred)
print(f"Accuracy with {acc:.4f}")

Accuracy with 1.0000


In [16]:
def run_one(run_id, X, y):
    e = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    with threadpool_limits(limits=1):
        e.fit(X, y)
    y_pred = e.predict(X)
    acc = accuracy_score(y, y_pred)
    return run_id, acc

n_runs = 10

results = Parallel(n_jobs=10)(
    delayed(run_one)(i, X, y) 
    for i in range(n_runs)
)

for run_id, acc in results:
    print(f"Run {run_id}: {acc:.4f}")

Run 0: 1.0000
Run 1: 1.0000
Run 2: 1.0000
Run 3: 1.0000
Run 4: 1.0000
Run 5: 1.0000
Run 6: 1.0000
Run 7: 1.0000
Run 8: 1.0000
Run 9: 1.0000


In [17]:
def run_one(run_id, X, y):
    e = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    e.fit(X, y)
    y_pred = e.predict(X)
    acc = accuracy_score(y, y_pred)
    return run_id, acc

n_runs = 10

results = Parallel(n_jobs=10)(
    delayed(run_one)(i, X, y) 
    for i in range(n_runs)
)

for run_id, acc in results:
    print(f"Run {run_id}: {acc:.4f}")

Run 0: 1.0000
Run 1: 1.0000
Run 2: 1.0000
Run 3: 1.0000
Run 4: 1.0000
Run 5: 1.0000
Run 6: 1.0000
Run 7: 1.0000
Run 8: 1.0000
Run 9: 1.0000
